# Gaussian-linear task with SMC-ABC

A lightweight SMC-ABC run is repeated on a small set of simulated observations. The resulting particle banks are checked with SBC and TARP.

In [ ]:
from pathlib import Path
import sys
import torch
import yaml
from sbi.inference import SMCABC

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT / "code" if (ROOT / "code" / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sfmpe.tasks.Gaussian import GaussianTask
from sfmpe.utils.metrics import check_sbc, check_tarp, run_sbc, run_tarp

with (ROOT / "configs" / "gaussian.yaml").open(encoding="utf-8") as stream:
    task = GaussianTask(yaml.safe_load(stream))
torch.manual_seed(37)

In [ ]:
num_observations = 8
num_particles = 40
thetas, observations = task.simulate_dataset((num_observations,))
particle_sets = []
for observation in observations:
    smc = SMCABC(
        simulator=task.simulate,
        prior=task.prior,
        show_progress_bars=False,
    )
    particles = smc(
        observation,
        num_particles=num_particles,
        num_initial_pop=100,
        num_simulations=600,
        epsilon_decay=0.5,
    )
    particle_sets.append(particles)
posterior_samples = torch.stack(particle_sets, dim=1)

ranks, dap_samples = run_sbc(
    thetas, posterior_samples, num_posterior_samples=num_particles
)
sbc_checks = check_sbc(
    ranks, task.sample_prior((num_observations,)), dap_samples,
    num_posterior_samples=num_particles,
)
ecp, alpha = run_tarp(thetas, posterior_samples, num_bins=4)
tarp_checks = check_tarp(ecp, alpha)
assert posterior_samples.shape == (num_particles, num_observations, task.theta_dim)
assert torch.isfinite(ecp).all()
print("SBC KS p-values:", sbc_checks["ks_pvals"])
print("TARP (area-to-curve, KS p-value):", tarp_checks)